In [2]:
import torch

import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 473

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")



Training on: cuda


# Import CIFAR-10 (in-distribution) dataset

In [12]:
# Import the batches here

# Note to self: These are hardcoded and apparently come from the actual CIFAR-10 dataset.
#	It's just to streamline the calculations instead of manually having to do it. I think the
#	tuple values are a (R,G,B) for the color channels.
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# Two versions of the training split are needed so validation has no augmentation.
train_data = datasets.CIFAR10(
    root="../data",
    train=True,
    download=False,
    transform=train_transform,
)
eval_data = datasets.CIFAR10(
    root="../data",
    train=True,
    download=False,
    transform=eval_transform,
)
test_data = datasets.CIFAR10(
    root="../data",
    train=False,
    download=False,
    transform=eval_transform,
)

print(f"Training images available: {len(train_data)}")
print(f"Test images: {len(test_data)}")
print(f"Classes: {train_data.classes}")

Training images available: 50000
Test images: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [13]:
split_generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(train_data), generator=split_generator)

val_indices = indices[:5000].tolist()
train_indices = indices[5000:].tolist()

train_loader = DataLoader(
    Subset(train_data, train_indices),
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=0,
)

val_loader = DataLoader(
    Subset(eval_data, val_indices),
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_data,
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

# Import SVHN (out-of-distribution) dataset

In [ ]:
# Import the batches here (street view house numbers)

# Import ResNet-20 model

In [14]:
# This hub repository is structured correctly and supports pretrained=False
model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=False)
model = model.to(device)  # move to the GPU

print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")

Using cache found in C:\Users\ellio/.cache\torch\hub\chenyaofo_pytorch-cifar-models_master


Total parameters: 272474


In [15]:
import torch.nn as nn
import torch.nn.functional as F

class SpectralNormConv2d(nn.Conv2d):
    def __init__(self, *args, norm_bound=6.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.norm_bound = norm_bound
        self.register_buffer("u", F.normalize(torch.randn(self.weight.shape[0]), dim=0))
        self.register_buffer("v", F.normalize(torch.randn(self.weight[0].numel()), dim=0))

    def forward(self, x):
        W = self.weight
        W_mat = W.reshape(W.shape[0], -1)
        if self.training:
            with torch.no_grad():
                v = F.normalize(W_mat.t() @ self.u, dim=0)
                u = F.normalize(W_mat @ v, dim=0)
                self.u.copy_(u); self.v.copy_(v)
        sigma = self.u @ (W_mat @ self.v)
        W = W * torch.clamp(self.norm_bound / sigma, max=1.0)
        return self._conv_forward(x, W, self.bias)

def to_sn(conv, c):
    new = SpectralNormConv2d(conv.in_channels, conv.out_channels, conv.kernel_size,
                             stride=conv.stride, padding=conv.padding,
                             bias=conv.bias is not None, norm_bound=c)
    new.weight.data.copy_(conv.weight.data)
    return new

def add_spectral_norm(model, c=6.0):
    for stage in (model.layer1, model.layer2, model.layer3):
        for block in stage:
            block.conv1 = to_sn(block.conv1, c)
            block.conv2 = to_sn(block.conv2, c)
    return model

sn_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=False)
sn_model = add_spectral_norm(sn_model, c=6.0).to(device)
print(sum(isinstance(m, SpectralNormConv2d) for m in sn_model.modules()))  # expect 18

18


Using cache found in C:\Users\ellio/.cache\torch\hub\chenyaofo_pytorch-cifar-models_master


In [16]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class RFFGaussianProcess(nn.Module):
    """Random-feature GP output layer (SNGP, Liu et al. 2020, Sec. 3.1)."""

    def __init__(self, in_features, num_classes, num_features=1024,
                 length_scale=2.0, ridge=1.0, use_input_norm=True):
        super().__init__()
        self.num_features = num_features
        self.num_classes = num_classes
        self.length_scale = length_scale
        self.ridge = ridge
        self.feature_scale = math.sqrt(2.0 / num_features)

        self.input_norm = nn.LayerNorm(in_features) if use_input_norm else None

        # Frozen random features (Eq. 7). Buffers are saved in state_dict,
        # so a reloaded model keeps the same random features.
        self.register_buffer("W_rff", torch.randn(num_features, in_features))
        self.register_buffer("b_rff", 2 * math.pi * torch.rand(num_features))

        # The only trainable weights: beta (Eq. 8)
        self.beta = nn.Linear(num_features, num_classes, bias=False)

        # Laplace posterior (Eq. 9): per-class precision sum and its inverse
        K, D = num_classes, num_features
        self.register_buffer("precision", torch.zeros(K, D, D))
        self.register_buffer("covariance", torch.eye(D).div(ridge).repeat(K, 1, 1))

    def features(self, h):
        if self.input_norm is not None:
            h = self.input_norm(h)
        h = h / self.length_scale                      # RBF length-scale
        return self.feature_scale * torch.cos(F.linear(h, self.W_rff, self.b_rff))

    def forward(self, h, return_var=False):
        phi = self.features(h)
        logits = self.beta(phi)                        # posterior mean logits
        if return_var:
            return logits, self.variance(phi)
        return logits                                  # plain tensor, so existing loops still work

    def variance(self, phi):
        # var_k(x) = phi^T Sigma_k phi, shape (B, K)
        return torch.einsum("bd,kde,be->bk", phi, self.covariance, phi)

    @torch.no_grad()
    def reset_precision(self):
        self.precision.zero_()

    @torch.no_grad()
    def update_precision(self, h):
        """Accumulate sum_i p_ik (1 - p_ik) phi_i phi_i^T for a batch of features h."""
        phi = self.features(h)
        p = F.softmax(self.beta(phi), dim=-1)          # (B, K)
        w = p * (1 - p)                                # (B, K)
        weighted = phi.unsqueeze(1) * w.unsqueeze(2)   # (B, K, D)
        self.precision.add_(torch.einsum("bkd,be->kde", weighted, phi))

    @torch.no_grad()
    def finalize_covariance(self):
        eye = torch.eye(self.num_features, device=self.precision.device, dtype=torch.float64)
        prec = self.precision.double() + self.ridge * eye
        self.covariance.copy_(torch.linalg.inv(prec).to(self.covariance.dtype))

    @torch.no_grad()
    def predict_proba(self, h, num_samples=10):
        """Algorithm 2: sample logits from N(mean, var), average the softmaxes."""
        phi = self.features(h)
        mean = self.beta(phi)
        std = self.variance(phi).clamp_min(0).sqrt()
        eps = torch.randn(num_samples, *mean.shape, device=mean.device)
        return F.softmax(mean + std * eps, dim=-1).mean(dim=0)

In [6]:
def build_sngp_model(norm_bound=6.0, gp_features=1024, length_scale=2.0, ridge=1.0):
	model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=False)
	model = add_spectral_norm(model, c=norm_bound)
	model.fc = RFFGaussianProcess(64, 10, num_features=gp_features, length_scale=length_scale, ridge=ridge)
	return model.to(device)

sngp_model = build_sngp_model()
print(sum(isinstance(x, SpectralNormConv2d) for x in sngp_model.modules()))
print(sum(p.numel() for p in sngp_model.parameters()))

Using cache found in C:\Users\ellio/.cache\torch\hub\chenyaofo_pytorch-cifar-models_master


18
282192


In [ ]:
## Set up training loss
criterion = torch.nn.CrossEntropyLoss()

epochs = 40 # number of iterations through the training batch
optimizer = torch.optim.SGD(
	model.parameters(),  # use the parameters (?) obtained by the model
	lr=0.1, # learning rate.
	momentum=0.9, # m
	weight_decay = 0.0005
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [22]:
# Train the model
def accuracy(model, loader):

	model.eval()
	correct = 0
	total = 0

	with torch.no_grad():  # why no gradient?
		for images, labels in loader:
			images = images.to(device)
			labels = labels.to(device)

			## Perform the softmax on the images
			predictions = model(images).argmax(dim=1)
			correct += (predictions == labels).sum().item()
			total += labels.size(0)

		return correct / total

def train(model, epochs=40, lr=0.1, output_path="output/cifar10_resnet_20_sngp.pt"):
	loss_function = nn.CrossEntropyLoss()
	opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
	scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
	best = 0.0
	for epoch in range(epochs):
		model.train()
		total_loss = 0
		seen = 0
		for images, labels in train_loader:
			images = images.to(device)
			labels = labels.to(device)

			opt.zero_grad()
			logits = model(images)
			loss = loss_function(logits, labels)
			loss.backward()
			opt.step()
			total_loss += loss.item() * labels.size(0)
			seen += labels.size(0)

		scheduler.step()
		val = accuracy(model, val_loader)
		print(f"epoch {epoch+1} / {epochs} loss = {total_loss/seen:.4f} val_acc={val:.4f}")
		if val > best:
			best = val

			output_path = Path("output/cifar10_resnet_20_sngp.pt")
			output_path.parent.mkdir(parents=True, exist_ok=True)
			best_val_accuracy = 0.0
			
			torch.save({"state_dict": model.state_dict(), "val_accuracy": val, "epoch": epoch+1}, output_path)

	print(f"Best validation accuracy: {best_val_accuracy}")
	print(f"Output saved to {output_path}")


# Note: get NLL and ECE

train(sngp_model)


epoch 1 / 40 loss = 0.6595 val_acc=0.7518
epoch 2 / 40 loss = 0.5308 val_acc=0.7172
epoch 3 / 40 loss = 0.5002 val_acc=0.7526
epoch 4 / 40 loss = 0.4860 val_acc=0.7262
epoch 5 / 40 loss = 0.4856 val_acc=0.7770
epoch 6 / 40 loss = 0.4697 val_acc=0.7828
epoch 7 / 40 loss = 0.4684 val_acc=0.7286
epoch 8 / 40 loss = 0.4607 val_acc=0.7292
epoch 9 / 40 loss = 0.4509 val_acc=0.8176
epoch 10 / 40 loss = 0.4498 val_acc=0.7916
epoch 11 / 40 loss = 0.4450 val_acc=0.8048
epoch 12 / 40 loss = 0.4434 val_acc=0.7860
epoch 13 / 40 loss = 0.4333 val_acc=0.7276
epoch 14 / 40 loss = 0.4218 val_acc=0.8332
epoch 15 / 40 loss = 0.4124 val_acc=0.8416
epoch 16 / 40 loss = 0.3992 val_acc=0.8340
epoch 17 / 40 loss = 0.3966 val_acc=0.7538
epoch 18 / 40 loss = 0.3899 val_acc=0.8196
epoch 19 / 40 loss = 0.3778 val_acc=0.8498
epoch 20 / 40 loss = 0.3680 val_acc=0.8172
epoch 21 / 40 loss = 0.3598 val_acc=0.8198
epoch 22 / 40 loss = 0.3377 val_acc=0.8498
epoch 23 / 40 loss = 0.3357 val_acc=0.8230
epoch 24 / 40 loss =

In [23]:
import time
import torch
from torch.utils.data import DataLoader, Subset

def get_features(m, x):
    # mirrors the hub model's forward, minus fc
    x = m.relu(m.bn1(m.conv1(x)))
    x = m.layer3(m.layer2(m.layer1(x)))
    return m.avgpool(x).flatten(1)

@torch.no_grad()
def fit_covariance(m, loader, log_every=50):
    m.eval()                       # BatchNorm uses running stats; spectral norm stops updating u, v
    m.fc.reset_precision()         # start from zero so reruns don't double count
    t0, n = time.time(), 0
    for i, (x, _) in enumerate(loader):
        m.fc.update_precision(get_features(m, x.to(device)))
        n += x.size(0)
        if (i + 1) % log_every == 0:
            print(f"  {n} images, {time.time() - t0:.1f}s")
    m.fc.finalize_covariance()     # Sigma = (ridge*I + precision)^-1, inverted in float64
    print(f"covariance fitted from {n} images in {time.time() - t0:.1f}s")

# Training subset, no augmentation (eval_data has eval_transform)
cov_loader = DataLoader(Subset(eval_data, train_indices), batch_size=256,
                        shuffle=False, num_workers=0)

# Fit on the exact weights you will evaluate
ckpt = torch.load("output/cifar10_resnet_20_sngp.pt", map_location=device)
sngp_model.load_state_dict(ckpt["state_dict"])   # adjust the key to match your checkpoint format
fit_covariance(sngp_model, cov_loader)

  12800 images, 1.7s
  25600 images, 2.9s
  38400 images, 4.2s
covariance fitted from 45000 images in 5.2s


In [24]:
fc = sngp_model.fc

# 1. get_features + fc must reproduce the normal forward pass
x, _ = next(iter(test_loader)); x = x.to(device)
with torch.no_grad():
    print("forward matches:", torch.allclose(fc(get_features(sngp_model, x)), sngp_model(x), atol=1e-5))

# 2. Math invariants
P = fc.precision
rel_asym = ((P - P.transpose(1, 2)).abs().max() / P.abs().max().clamp_min(1e-12)).item()
diag = fc.covariance.diagonal(dim1=1, dim2=2)               # (K, D)
print(f"precision asymmetry (relative): {rel_asym:.2e}")      # expect ~1e-6 or smaller
print(f"Sigma diag  min={diag.min().item():.4f}  mean={diag.mean().item():.4f}  max={diag.max().item():.4f}"
      f"   (prior = {1 / fc.ridge})")                          # max must not exceed the prior

# 3. How many directions did the training data actually constrain? (class 0 as a sample)
eig = torch.linalg.eigvalsh(P[0].double())
print("directions where data evidence exceeds the prior:", (eig > fc.ridge).sum().item(), "of", fc.num_features)

# 4. Held-out data should look less uncertain than noise
@torch.no_grad()
def mean_variance(m, x):
    _, var = m.fc(get_features(m, x), return_var=True)
    return var.mean().item()

xv, _ = next(iter(val_loader)); xv = xv.to(device)
print("mean variance, val images:", mean_variance(sngp_model, xv))
print("mean variance, noise     :", mean_variance(sngp_model, torch.randn_like(xv)))

torch.save({"state_dict": sngp_model.state_dict()}, "output/cifar10_resnet_20_sngp_final.pt")

forward matches: True
precision asymmetry (relative): 2.20e-07
Sigma diag  min=0.8398  mean=0.9535  max=0.9853   (prior = 1.0)
directions where data evidence exceeds the prior: 23 of 1024
mean variance, val images: 0.1434534639120102
mean variance, noise     : 0.34238773584365845


In [ ]:
import math
import numpy as np
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score   # pip install scikit-learn if missing

N_OOD = 10_000   # balances the 10,000 CIFAR test images; set to None for all 26,032
svhn_test = datasets.SVHN(root="../data", split="test", download=True, transform=eval_transform)
if N_OOD is not None:
    g = torch.Generator().manual_seed(SEED)
    svhn_test = Subset(svhn_test, torch.randperm(len(svhn_test), generator=g)[:N_OOD].tolist())
svhn_loader = DataLoader(svhn_test, batch_size=256, shuffle=False, num_workers=0)
print("SVHN images:", len(svhn_test))

In [ ]:
K = 10

def nll(probs, labels):
    return F.nll_loss(probs.clamp_min(1e-12).log(), labels).item()


def calc_ECE(probs, labels, n_bins=15):
	conf, pred = probs.max(1)
	correct = (pred == labels).float()
	edges = torch.linspace(0, 1, n_bins + 1)
	total = 0.0
	for low, high in zip(edges[:-1], edges[1:]):
		in_bin = (conf > low) & (conf <= high)
		if in_bin.any():
			total += in_bin.float().mean().item() * abs(correct[in_bin].mean().item() - conf[in_bin].mean().item())
	return total

def ds_uncertainty(logits):
	return torch.sigmoid(math.log(K) - torch.logsumexp(logits, dim=1))

def ood_scores(s_in, s_ood):
    y = np.r_[np.zeros(len(s_in)), np.ones(len(s_ood))]      # SVHN = positive class
    s = np.r_[s_in.numpy(), s_ood.numpy()]
    return roc_auc_score(y, s), average_precision_score(y, s)

@torch.no_grad()
def collect(m, loader, sngp, num_samples=10):
    m.eval()
    probs, logits, labels, variances = [], [], [], []
    for x, y in loader:
        x = x.to(device)
        if sngp:
            h = get_features(m, x)
            logits.append(m.fc(h).cpu())                                    # mean logits
            probs.append(m.fc.predict_proba(h, num_samples).cpu())          # MC-averaged softmax
            variances.append(m.fc.variance(m.fc.features(h)).mean(1).cpu()) # avg over classes
        else:
            lg = m(x)
            logits.append(lg.cpu())
            probs.append(F.softmax(lg, dim=-1).cpu())
        labels.append(y)
    return dict(probs=torch.cat(probs), logits=torch.cat(logits), labels=torch.cat(labels),
                var=torch.cat(variances) if sngp else None)

def evaluate(name, m, sngp):
    torch.manual_seed(SEED)                     # fixes the MC samples in predict_proba
    r_in, r_ood = collect(m, test_loader, sngp), collect(m, svhn_loader, sngp)
    p, y = r_in["probs"], r_in["labels"]
    row = {"model": name, "acc": (p.argmax(1) == y).float().mean().item(),
           "NLL": nll(p, y), "ECE": calc_ECE(p, y)}
    pairs = {
        "maxp": (1 - r_in["probs"].max(1).values, 1 - r_ood["probs"].max(1).values),
        "DS":   (ds_uncertainty(r_in["logits"]), ds_uncertainty(r_ood["logits"])),
        "var":  (r_in["var"], r_ood["var"]) if sngp else None,
    }
    for sname, pair in pairs.items():
        auroc, aupr = ood_scores(*pair) if pair else (float("nan"), float("nan"))
        row[f"ROC_{sname}"], row[f"PR_{sname}"] = auroc, aupr
    return row

def show(rows):
    cols = list(rows[0].keys())
    print(" ".join(f"{c:>10}" for c in cols))
    for r in rows:
        print(" ".join(f"{v:>10.4f}" if isinstance(v, float) else f"{v:>10}" for v in r.values()))